# TF-IDF (Term Frequency — Inverse Document Frequency)

**Goal:** convert text documents into weighted numerical vectors where important words score higher and common words are down-weighted.

**Flow:** Data → Vectorizer → fit_transform → Sparse TF-IDF matrix → Dense array → Vocabulary

**How it differs from BoW:** BoW counts raw word frequencies. TF-IDF adjusts those counts — words appearing in many documents (like "the", "is") get lower weight, while rare but informative words get higher weight.

## Step 1: Import tools

- `TfidfVectorizer` from sklearn handles tokenization, vocabulary building, TF-IDF weighting, and L2 normalization — all in one object.

**Why not separate TF and IDF manually?** You could compute term frequency and inverse document frequency with loops and dicts, but `TfidfVectorizer` is optimized, battle-tested, and produces sparse output out of the box.

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

## Step 2: Define sample data

- `data` is the raw input corpus — same documents used in OHE and BoW notebooks.
- No manual preprocessing needed here — `TfidfVectorizer` handles lowercasing and tokenization internally by default.

**Why no cleaning step?** Unlike the BoW notebook where we manually lemmatized, this example uses the vectorizer's built-in defaults. For production use, you'd still want to add lemmatization or pass a custom `tokenizer`.

In [2]:
data =["SDE loves coding",
    "SDE loves AI",
    "SDE codes in Python",
    "SDE codes AI"]

## Step 3: Initialize the vectorizer

- `TfidfVectorizer()` with default settings: lowercases text, splits on word boundaries, applies TF-IDF weighting with L2 normalization.
- Internally it computes: `tfidf(t, d) = tf(t, d) × idf(t)` then normalizes each document vector to unit length.

**Why L2 normalization by default?** It ensures documents of different lengths produce comparable vectors. Without it, longer documents would dominate similarity calculations.

**Alternative:** use `CountVectorizer` + `TfidfTransformer` as a two-step pipeline for more granular control over each stage.

In [3]:
vectorizer = TfidfVectorizer()

## Step 4: Fit and transform the data

- `fit_transform(data)` learns the vocabulary + IDF weights and transforms documents into TF-IDF vectors in one pass.
- The first call below is exploratory (result not stored). The second stores the sparse matrix in `tf_idf_vec`.

**Why call `fit_transform` twice?** The first is a quick check; the second stores the result. In practice, you'd call it once. Both produce identical output since the data hasn't changed.

In [4]:
# Quick check — output not stored, just to see the sparse repr
vectorizer.fit_transform(data)

<4x7 sparse matrix of type '<class 'numpy.float64'>'
	with 13 stored elements in Compressed Sparse Row format>

In [5]:
# Store the sparse TF-IDF matrix for inspection
tf_idf_vec = vectorizer.fit_transform(data)

## Step 5: Inspect the TF-IDF matrix

- The raw object is a **sparse CSR matrix** — memory-efficient for large vocabularies.
- `.toarray()` converts it to a dense NumPy array for visual inspection.
- The loop prints each document's vector row-by-row for readability.

**Why are values between 0 and 1?** L2 normalization scales each row so its Euclidean length = 1. Higher values mean the word is more important to that specific document relative to the corpus.

**Why sparse?** Same reason as BoW — most entries are zero. A 50k-word vocabulary with 15 words per document means ~99.97% zeros.

In [6]:
# Sparse repr: shows (doc_index, word_index) → tfidf_score for non-zero entries
tf_idf_vec

<4x7 sparse matrix of type '<class 'numpy.float64'>'
	with 13 stored elements in Compressed Sparse Row format>

In [7]:
# Dense view — each row is a document, each column is a vocabulary word
tf_idf_vec.toarray()

array([[0.        , 0.        , 0.72664149, 0.        , 0.5728925 ,
        0.        , 0.37919167],
       [0.64043405, 0.        , 0.        , 0.        , 0.64043405,
        0.        , 0.42389674],
       [0.        , 0.46345796, 0.        , 0.58783765, 0.        ,
        0.58783765, 0.30675807],
       [0.64043405, 0.64043405, 0.        , 0.        , 0.        ,
        0.        , 0.42389674]])

In [8]:
# Print each document's TF-IDF vector on its own line
for i, vec in enumerate(tf_idf_vec.toarray()):
    print(f"Doc {i}: {vec}")

Doc 0: [0.         0.         0.72664149 0.         0.5728925  0.
 0.37919167]
Doc 1: [0.64043405 0.         0.         0.         0.64043405 0.
 0.42389674]
Doc 2: [0.         0.46345796 0.         0.58783765 0.         0.58783765
 0.30675807]
Doc 3: [0.64043405 0.64043405 0.         0.         0.         0.
 0.42389674]


## Step 6: Inspect the vocabulary

- `get_feature_names_out()` returns the learned vocabulary in alphabetical order.
- Each column index in the TF-IDF matrix maps to the word at the same position in this array.

**Alternative:** access `vectorizer.vocabulary_` for a `{word: index}` dict, useful when you need to look up a specific word's column position.

In [9]:
vectorizer.get_feature_names_out()

array(['ai', 'codes', 'coding', 'in', 'loves', 'python', 'sde'],
      dtype=object)

- **TF-IDF** = Term Frequency × Inverse Document Frequency. Words common across all documents get low weight; rare words get high weight.
- `TfidfVectorizer` combines `CountVectorizer` + `TfidfTransformer` into one step.
- Output is L2-normalized by default — each document vector has unit length, making cosine similarity straightforward.
- The result is a **sparse CSR matrix** — convert with `.toarray()` for inspection.
- `fit_transform()` learns vocabulary + IDF weights and transforms in one pass. Use `transform()` alone for new data.
- Like BoW, TF-IDF ignores word order. `"dog bites man"` and `"man bites dog"` produce the same vector.

**Key formula:**
- `tf(t, d)` = count of term `t` in document `d`
- `idf(t)` = log((1 + n) / (1 + df(t))) + 1, where `n` = total docs, `df(t)` = docs containing `t`
- sklearn adds smoothing (+1) to avoid division by zero and zero IDF.

**Limitations:** still no word order, no semantics, vocabulary-dependent.


1. **"How is TF-IDF different from Bag of Words?"**
   BoW gives raw word counts. TF-IDF re-weights those counts — common words (appearing in many docs) get penalized via IDF, making rare but informative words stand out.

2. **"What happens if a word appears in every document?"**
   Its IDF approaches zero (with smoothing, it gets a very low weight). The word is treated as uninformative — similar to a stopword being automatically down-weighted.

3. **"Why does sklearn add +1 smoothing in the IDF formula?"**
   To prevent division by zero when a term appears in all documents, and to avoid completely zeroing out common terms. It's `log((1+n)/(1+df)) + 1`.

4. **"What does L2 normalization do here?"**
   It scales each document vector to unit length (Euclidean norm = 1). This makes cosine similarity between documents equivalent to a dot product, and prevents longer documents from dominating.

5. **"Can TF-IDF handle unseen words at test time?"**
   No. Words not in the training vocabulary are silently ignored by `transform()`. The vocabulary is frozen after `fit()`.

6. **"Why is the output sparse?"**
   Each document uses only a handful of words from the full vocabulary. Sparse CSR format stores only non-zero values — critical for corpora with vocabularies in the tens of thousands.

7. **"When would you choose TF-IDF over word embeddings?"**
   TF-IDF works well for information retrieval, keyword extraction, and when you need interpretable features. Embeddings (Word2Vec, BERT) are better for capturing semantics but are opaque and heavier.

8. **"What's the difference between `TfidfVectorizer` and `CountVectorizer` + `TfidfTransformer`?"**
   Functionally identical. The two-step approach gives you access to raw counts before TF-IDF weighting — useful if you need both representations.

1. **"Two documents use different words but mean the same thing. How does TF-IDF handle this?"**
   Poorly. `"happy"` and `"joyful"` are treated as completely separate features with zero overlap. TF-IDF has no notion of synonymy — you'd need embeddings or a thesaurus-based approach.

2. **"How would you modify this to capture phrases, not just single words?"**
   Use `ngram_range=(1, 2)` or `(1, 3)` in `TfidfVectorizer`. This adds bigrams/trigrams (e.g., `"machine learning"`) as features alongside unigrams. Trade-off: vocabulary size explodes.

3. **"If you add 10,000 new documents, do you need to refit?"**
   Yes — the vocabulary and IDF weights are learned from the training set. New documents may contain unseen words. You'd call `fit_transform()` on the full updated corpus or use `HashingVectorizer` for a stateless alternative.

4. **"Why do some TF-IDF values look the same across different documents?"**
   If two documents share a word that has the same TF and the document lengths are similar after normalization, the TF-IDF scores will be close or identical. The IDF component is constant per word across all documents.

5. **"How would you use TF-IDF vectors for classification?"**
   Feed them directly into a classifier — `LogisticRegression`, `SVM`, or `NaiveBayes` all work well with sparse TF-IDF input. This is a classic text classification pipeline and often a strong baseline before trying deep learning.